In [1]:
from pathlib import Path
import json
import pandas as pd

PROCESSED_DIR = Path(
    r"C:\Users\user\OneDrive - University of Stirling\Desktop\dissertation\mobility-representative-profile\data\demographic\processed"
)

print("Folder exists:", PROCESSED_DIR.exists())
print("Path:", PROCESSED_DIR)

Folder exists: True
Path: C:\Users\user\OneDrive - University of Stirling\Desktop\dissertation\mobility-representative-profile\data\demographic\processed


In [2]:
json_files = sorted(PROCESSED_DIR.rglob("*.json"))

print(f"Found {len(json_files)} JSON files:\n")

for path in json_files:
    print("-", path.relative_to(PROCESSED_DIR))

Found 6 JSON files:

- age\age.json
- car_ownership\car_ownership.json
- economic_activity_status\economic_activity_status.json
- gender\gender.json
- household_by_age\household_by_age.json
- industry\industry_by_age.json


In [3]:
loaded_json = {}

for path in json_files:
    dataset_name = path.parent.name

    try:
        with path.open("r", encoding="utf-8") as file:
            data = json.load(file)

        loaded_json[dataset_name] = data

        print(
            f"PASS | {dataset_name:25} "
            f"| type: {type(data).__name__:8} "
            f"| length: {len(data)}"
        )

    except json.JSONDecodeError as error:
        print(
            f"FAIL | {dataset_name:25} "
            f"| Invalid JSON: {error}"
        )

    except Exception as error:
        print(
            f"FAIL | {dataset_name:25} "
            f"| {type(error).__name__}: {error}"
        )

PASS | age                       | type: list     | length: 9
PASS | car_ownership             | type: list     | length: 1
PASS | economic_activity_status  | type: list     | length: 63
PASS | gender                    | type: list     | length: 18
PASS | household_by_age          | type: list     | length: 135
PASS | industry                  | type: list     | length: 171


In [4]:
for dataset_name, data in loaded_json.items():
    print(f"\n{'=' * 60}")
    print(dataset_name.upper())
    print(f"{'=' * 60}")

    if len(data) > 0:
        print("Keys:", list(data[0].keys()))
        print("First record:")
        print(data[0])


AGE
Keys: ['Lower tier local authorities', 'Age Group', 'Observation']
First record:
{'Lower tier local authorities': 'Manchester', 'Age Group': '0-4', 'Observation': 34378}

CAR_OWNERSHIP
Keys: ['local authority: district / unitary (as of April 2023)', 'Total: All households', 'No cars or vans in household', '1 car or van in household', '2 cars or vans in household', '3 or more cars or vans in household']
First record:
{'local authority: district / unitary (as of April 2023)': 'Manchester', 'Total: All households': 214732.0, 'No cars or vans in household': 83770.0, '1 car or van in household': 92267.0, '2 cars or vans in household': 31611.0, '3 or more cars or vans in household': 7084.0}

ECONOMIC_ACTIVITY_STATUS
Keys: ['Lower tier local authorities', 'Economic activity status (7 categories)', 'Age Group', 'Observation']
First record:
{'Lower tier local authorities': 'Manchester', 'Economic activity status (7 categories)': 'Does not apply', 'Age Group': '0-4', 'Observation': 34378}



In [5]:
age_df = pd.DataFrame(loaded_json["age"])
gender_df = pd.DataFrame(loaded_json["gender"])

gender_totals = (
    gender_df
    .groupby("Age Group", as_index=False)["Observation"]
    .sum()
    .rename(columns={"Observation": "Gender Total"})
)

age_gender_check = age_df.merge(
    gender_totals,
    on="Age Group",
    how="left"
)

age_gender_check["Difference"] = (
    age_gender_check["Observation"]
    - age_gender_check["Gender Total"]
)

display(
    age_gender_check[
        ["Age Group", "Observation", "Gender Total", "Difference"]
    ]
)

,Age Group,Observation,Gender Total,Difference
0,0-4,34378,34378,0
1,5-11,51349,51349,0
2,12-17,41226,41225,1
3,18-22,61780,61783,-3
4,23-34,123440,123441,-1
5,35-49,108897,108892,5
6,50-64,78705,78703,2
7,65-74,30299,30300,-1
8,75+,21868,21869,-1


In [8]:
TOLERANCE = 10

assert age_gender_check["Difference"].abs().le(TOLERANCE).all(), \
    f"Gender totals differ from age totals by more than {TOLERANCE} people."

print(
    f"PASS: Gender totals match age totals "
    f"within a tolerance of ±{TOLERANCE} people."
)

PASS: Gender totals match age totals within a tolerance of ±10 people.


In [9]:
age_gender_check["Relative Difference"] = (
    age_gender_check["Difference"].abs()
    / age_gender_check["Observation"]
)

assert age_gender_check["Relative Difference"].le(0.001).all(), \
    "Gender totals differ from age totals by more than 0.1%."

print("PASS: All relative differences are below 0.1%.")

PASS: All relative differences are below 0.1%.


## Validation Summary

All six processed demographic JSON files were successfully loaded and
validated.

- All files were valid and non-empty.
- Required demographic and category fields were retained.
- Percentage fields were excluded from processed outputs so that
  probabilities can be calculated consistently during integration.
- Gender totals were compared against the age totals.
- Minor discrepancies were within the predefined absolute and relative
  tolerances and were retained without manual adjustment.
- Car ownership was retained as a household-level, wide-format dataset.